# 55. Excel Files — `openpyxl · xlsxwriter`
**Python Libraries Lab · Phase 3 · Files & Automation**

_Create, style, read and update Excel workbooks from Python._

✅ **Runs in Colab** as-is.

Much of the business world runs on Excel. Python can create and edit real Excel files, with formatting, formulas, charts and several sheets, so a report that someone builds by hand every Monday can be produced in seconds. openpyxl reads and writes existing workbooks. XlsxWriter only writes new files, but has excellent charts and formatting, and pandas can use either one.

**In this notebook:** You will build the ShopEasy monthly sales workbook. A styled sheet with totals as formulas, a pandas export with a chart and coloured cells using XlsxWriter, then you will read a workbook back, update it and add a new sheet with openpyxl.

**How to use:** run each cell in order with **Shift + Enter**, or use **Runtime → Run all**. Read the short note above each cell first, then change the code and run it again: that is how you learn.

## Setup
The next cell makes sure the library is installed (Colab already has most of them; it takes a few seconds).

In [ ]:
!pip install -q openpyxl xlsxwriter
import importlib.metadata as md
for pkg in ['openpyxl', 'xlsxwriter']:
    print(pkg, md.version(pkg))

## Step 1 · A styled sheet with formulas (openpyxl)
Write rows, then a total row using Excel formulas. Style the header and format numbers as rupees.

In [ ]:
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

wb = Workbook()
ws = wb.active
ws.title = "January"
rows = [("City", "Orders", "Revenue", "Avg order"),
        ("Pune", 169, 660869), ("Mumbai", 146, 569185), ("Delhi", 137, 533984), ("Bengaluru", 148, 463763)]
for r in rows:
    ws.append(r)
for i in range(2, 6):
    ws[f"D{i}"] = f"=C{i}/B{i}"
ws.append(["Total", "=SUM(B2:B5)", "=SUM(C2:C5)", "=C6/B6"])

header = PatternFill("solid", fgColor="2C5A9A")
for cell in ws[1]:
    cell.font = Font(bold=True, color="FFFFFF"); cell.fill = header; cell.alignment = Alignment(horizontal="center")
for cell in ws[6]:
    cell.font = Font(bold=True); cell.border = Border(top=Side(style="thin"))
for row in ws.iter_rows(min_row=2, min_col=3, max_col=4):
    for cell in row:
        cell.number_format = '"Rs."#,##0'
for col in range(1, 5):
    ws.column_dimensions[get_column_letter(col)].width = 14
ws.freeze_panes = "A2"
wb.save("shopeasy_january.xlsx")
print("Saved shopeasy_january.xlsx. Formula in D6:", ws["D6"].value)

## Step 2 · pandas + XlsxWriter: charts and colours
Export a DataFrame, then add a chart and conditional formatting with XlsxWriter.

In [ ]:
import pandas as pd, numpy as np
rng = np.random.default_rng(1)
daily = pd.DataFrame({"date": pd.date_range("2026-01-01", periods=31), "orders": rng.integers(15, 40, 31)})
daily["revenue"] = daily["orders"] * rng.integers(800, 1800, 31)

with pd.ExcelWriter("shopeasy_daily.xlsx", engine="xlsxwriter") as xw:
    daily.to_excel(xw, sheet_name="Daily", index=False)
    book, sheet = xw.book, xw.sheets["Daily"]
    money = book.add_format({"num_format": '"Rs."#,##0'})
    sheet.set_column("A:A", 12, book.add_format({"num_format": "dd mmm"}))
    sheet.set_column("C:C", 14, money)
    sheet.conditional_format("C2:C32", {"type": "3_color_scale"})
    chart = book.add_chart({"type": "line"})
    chart.add_series({"name": "Revenue", "categories": ["Daily", 1, 0, 31, 0], "values": ["Daily", 1, 2, 31, 2]})
    chart.set_title({"name": "January revenue"}); chart.set_legend({"none": True})
    sheet.insert_chart("E2", chart, {"x_scale": 1.4})
print("Saved shopeasy_daily.xlsx with a chart")

## Step 3 · Read a workbook back
`data_only=False` shows formulas; values written by Python are read directly.

In [ ]:
wb = load_workbook("shopeasy_january.xlsx")
ws = wb["January"]
print("Sheets:", wb.sheetnames, "| size:", ws.dimensions)
for row in ws.iter_rows(min_row=1, max_row=6, values_only=True):
    print(row)
print(pd.read_excel("shopeasy_daily.xlsx").head(3))

## Step 4 · Update and add a sheet
Change a value, insert a row, add a summary sheet, save as a new file.

In [ ]:
ws["B3"] = 150                                  # corrected Mumbai orders
ws.insert_rows(6)
ws["A6"], ws["B6"], ws["C6"] = "Chennai", 98, 312400
ws["D6"] = "=C6/B6"
ws["B7"], ws["C7"], ws["D7"] = "=SUM(B2:B6)", "=SUM(C2:C6)", "=C7/B7"
summary = wb.create_sheet("Summary", 0)
summary["A1"], summary["B1"] = "Best city", "=INDEX(January!A2:A6, MATCH(MAX(January!C2:C6), January!C2:C6, 0))"
summary["A2"], summary["B2"] = "Generated by", "Python + openpyxl"
wb.save("shopeasy_january_v2.xlsx")
print(load_workbook("shopeasy_january_v2.xlsx").sheetnames)

## ✍️ Your turn
Open `shopeasy_january_v2.xlsx` and store the **value of cell A6** on the `January` sheet in `answer`.

Write your code in the next cell, run it, then run the check cell.

In [ ]:
answer = None  # your code here

In [ ]:
def check():
    return answer == "Chennai", f"Expected 'Chennai', got {answer!r}"

# ▶ Run this cell to check your answer
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

### Stuck?
The next cell holds a solution. In Colab its code is hidden: double-click the cell to see it, or just run it.

In [ ]:
# @title 👀 Solution (double-click to show the code) { display-mode: "form" }
answer = load_workbook("shopeasy_january_v2.xlsx")["January"]["A6"].value

# Check again with the solution
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

## Recap
- **Workbook / worksheet**
- **Cell / range**
- **Formula**
- **Number format**
- **Conditional format**
- **Chart**

**Next:** 56. PDF Files (`reportlab · pypdf`): [open in Colab](https://colab.research.google.com/github/aashaysocial27-maker/python-libraries-colab/blob/main/notebooks/56_reportlab_pypdf.ipynb)

## 💻 Run it on your own computer
1. Install Python 3.10+ from python.org (Windows: tick **Add python.exe to PATH**).
2. Open a terminal (Windows: *Command Prompt*; Mac: *Terminal*) and run:
```bash
mkdir python-lab && cd python-lab
python -m venv .venv
# Windows:  .venv\Scripts\activate      Mac/Linux:  source .venv/bin/activate
pip install openpyxl xlsxwriter
pip install notebook
jupyter notebook
```
3. In Colab choose **File → Download → Download .ipynb**, then open that file in the Jupyter page that opened in your browser.

Lines that start with `!` or `%%` are notebook commands. In a plain terminal, type `!` commands without the `!`.